In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/features_v1.csv")
df.head()

,qualifyingPosition,pitStopCount,driver_form_avg,constructor_form_avg,circuitType_street,finishPosition,avgLapTime_s,constructorPoints
0,NaN,0.0,1.8,NaN,False,13,NaN,0.0
1,NaN,0.0,1.8,NaN,False,13,NaN,0.0
2,NaN,0.0,1.8,NaN,False,13,NaN,0.0
3,NaN,0.0,1.8,NaN,False,13,NaN,0.0
4,NaN,0.0,1.8,NaN,False,13,NaN,0.0


In [9]:
features = [
    "qualifyingPosition",
    "pitStopCount",
    "driver_form_avg",
    "constructor_form_avg",
    "circuitType_street"
]

In [12]:
df_lt = df.dropna(subset=features + ["avgLapTime_s"]).copy()

X_lt = df_lt[features]
y_lt = df_lt["avgLapTime_s"]

In [13]:
from sklearn.model_selection import train_test_split

X_train_lt, X_test_lt, y_train_lt, y_test_lt = train_test_split(
    X_lt, y_lt, test_size=0.2, random_state=42
)

In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lr_lt = LinearRegression()
lr_lt.fit(X_train_lt, y_train_lt)

y_pred_lt = lr_lt.predict(X_test_lt)

print("Avg Lap Time — Linear Regression (Features)")
print("MAE:", mean_absolute_error(y_test_lt, y_pred_lt))
print("RMSE:", np.sqrt(mean_squared_error(y_test_lt, y_pred_lt)))
print("R²:", r2_score(y_test_lt, y_pred_lt))

Avg Lap Time — Linear Regression (Features)
MAE: 14.45310965132544
RMSE: 23.97523796386947
R²: 0.023775499156457047


Even though LR performs poorly for lap time:

It demonstrates the limitations of linear assumptions
It’s easy to implement
It gives a clear contrast with Random Forest

Linear Regression is used as a first, interpretable baseline. Its poor performance (R² ≈ 0.02) highlights that average lap time depends on non-linear interactions and circuit-specific effects. Random Forest captures these effects much more effectively.

In [15]:
from sklearn.ensemble import RandomForestRegressor

rf_lt = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_lt.fit(X_train_lt, y_train_lt)
y_pred_rf_lt = rf_lt.predict(X_test_lt)

print("Avg Lap Time — Random Forest (Features)")
print("MAE:", mean_absolute_error(y_test_lt, y_pred_rf_lt))
print("RMSE:", np.sqrt(mean_squared_error(y_test_lt, y_pred_rf_lt)))
print("R²:", r2_score(y_test_lt, y_pred_rf_lt))

Avg Lap Time — Random Forest (Features)
MAE: 4.174628618399396
RMSE: 14.781844040607545
R²: 0.6289084356871573


In [16]:
import pandas as pd

importance = pd.Series(
    rf_lt.feature_importances_,
    index=features
).sort_values(ascending=False)

importance

driver_form_avg         0.327124
constructor_form_avg    0.311145
qualifyingPosition      0.265400
pitStopCount            0.082687
circuitType_street      0.013644
dtype: float64

Success DOES NOT mean high R²
This is motorsport — high randomness.

Success means:
Features improve relative performance
Results make domain sense